# Environment

In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import re

import data_preprocessing.kbase as kb 

from pathlib import Path
from typing import List, Tuple, Dict

import matplotlib.pyplot as plt

import os

# Functions

# Data Loading

## EDATOSCASO-LLAMADAS

In [ ]:
# Loading exported table
df_triaje = pq.read_table(os.path.join(kb.source_tables_path, "tabla_madre/master_table_4demands.parquet")).to_pandas()
df_triaje.shape

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(5)


# Run Pipeline in subet

In [ ]:
# Flag to test the pipeline in a subset of 100K
## must be either "all", to run in all dataframe, or "test", to run it in the first 100K rows
df_triaje = kb.df_pipeline_test(df_triaje, "all").sort_values(by="DEMANDAPK")

# Column Reduction


In [ ]:
# IDTIPODEMANDA1 and IDTIPODEMANDA2: additional information outside of the scope

# Columns to drop
columns_to_drop = ['IDTIPODEMANDA2', 'IDTIPODEMANDA3'] 

# Drop columns
df_triaje = df_triaje.drop(columns = columns_to_drop)
# df_triaje.info()

# Data cleaning

## Rename columns

In [ ]:
df_triaje = kb.rename_columns(df_triaje)


In [ ]:
df_triaje.head(1)

## FECHADEMANDA - demand_date

In [ ]:
# Convert the raw string date
df_triaje["demand_date"] = pd.to_datetime(
    df_triaje["demand_date"],
    format="%Y%m%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "demand_date" rows: {df_triaje["demand_date"].isna().sum()}')

df_triaje["demand_date"].dtype



## FNACIMIENTO - birth_date

In [ ]:
# Convert the raw string date
df_triaje["birth_date"] = pd.to_datetime(
    df_triaje["birth_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Count how many failed conversions 
print(f'Number of invalid "birth_date" rows: {df_triaje["birth_date"].isna().sum()}')

df_triaje["birth_date"].dtype

## EDAD - age

In [ ]:
df_triaje["age_type"].value_counts(dropna = False)

In [ ]:
df_triaje["age"].value_counts(dropna=False).sort_index(ascending=True)

In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()



In [ ]:

def process_age_pipeline(df, birth_col="birth_date", demand_col="demand_date",
                         age_col="age", age_type_col="age_type"):
    """
    Full age processing pipeline applied directly on `age` column.

    Steps:
    1. Compute age from birth_date when available.
    2. Compute age from reported age + age_type (A/M/D/S) when birth_date is missing.
    3. Replace the original 'age' column with the best available estimate.
    4. Round age to nearest integer.
    5. Filter ages to keep only values between 12 and 100.
    6. Convert final age to Int8.
    7. Remove auxiliary processing columns.

    Returns:
        The dataframe with an updated 'age' column (Int8).
    """

    # ------------------------------------------------------------
    # Step 1 — Compute age in years using birth_date
    # ------------------------------------------------------------
    df["age_from_birthdate"] = (
        (df[demand_col] - df[birth_col])
        .dt.total_seconds() / (365.25 * 24 * 3600)
    )

    # ------------------------------------------------------------
    # Step 2 — Define conversion multipliers for age_type
    # ------------------------------------------------------------
    unit_to_years = {
        "A": 1.0,        # years
        "M": 1/12,       # months
        "D": 1/365.25,   # days
        "S": 1/52,       # weeks
    }

    # ------------------------------------------------------------
    # Step 3 — Normalize age_type into uppercase strings
    # ------------------------------------------------------------
    df[age_type_col] = df[age_type_col].astype("string").str.upper()

    # ------------------------------------------------------------
    # Step 4 — Compute reported age in years using age and age_type
    # ------------------------------------------------------------
    df["reported_age_years"] = df[age_col] * df[age_type_col].map(unit_to_years)

    # ------------------------------------------------------------
    # Step 5 — Mask for rows missing birth_date but having reported age
    # ------------------------------------------------------------
    mask_missing_birth = (
        df[birth_col].isna() &
        df[age_col].notna() &
        (df[age_col] != 0)
    )

    # ------------------------------------------------------------
    # Step 6 — Update age directly
    # ------------------------------------------------------------
    # Start with age_from_birthdate when available
    df[age_col] = df["age_from_birthdate"]

    # Replace missing birthdate ages with reported_age_years
    df.loc[mask_missing_birth, age_col] = df.loc[mask_missing_birth, "reported_age_years"]

    # ------------------------------------------------------------
    # Step 7 — Round final age to nearest integer
    # ------------------------------------------------------------
    df[age_col] = df[age_col].round()

    # ------------------------------------------------------------
    # Step 8 — Convert to Int16
    # ------------------------------------------------------------
    df[age_col] = df[age_col].astype("Int16")

    # ------------------------------------------------------------
    # Step 9 — Remove auxiliary columns
    # ------------------------------------------------------------
    df = df.drop(columns=["age_from_birthdate", "reported_age_years"])

    return df

df_triaje = process_age_pipeline(df_triaje)


In [ ]:
mask = (
    df_triaje["birth_date"].isna() &             # birth_date es NaT
    df_triaje["age"].notna() &              # call_age NO es NaN
    (df_triaje["age"] != 0)                 # call_age NO es 0
)

mask.sum()

In [ ]:
## Exclusion of outliers and filtering by age range of interest
nrows_before_age_exclusion = len(df_triaje)
print(f"Number of rows before age filtering: {nrows_before_age_exclusion}")

df_triaje = df_triaje[df_triaje["age"].between(12, 100)]

nrows_after_age_exclusion = len(df_triaje)
print(f"Number of rows after age filtering: {nrows_after_age_exclusion}")

print("Rows removed after excluding outliers and filtering by age range:",
      nrows_before_age_exclusion - nrows_after_age_exclusion)


In [ ]:
df_triaje['age'].hist(bins=20, range=(0, 100), color='skyblue', edgecolor='black')
plt.title('Histogram of Age')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6) 
plt.show()

df_triaje.boxplot(column=['age'], by='demand_type_1')
plt.title('Boxplot of Age')
plt.ylabel('Age')
plt.show()


## SEX - sex

In [ ]:
print("Value counts of 'sex' column before conversion:")
print(df_triaje['sex'].value_counts(dropna=False))

In [ ]:
# Map dictionary
## 'D' and None values are unknown
## Male (H): 0
## Female (M): 1
map_sex = {
    'D': np.nan,
    'None': np.nan,
    'H': 0,
    'M': 1
}

# Map the values
df_triaje["sex"] = df_triaje["sex"].map(map_sex)

# Convert data type
df_triaje["sex"] = pd.to_numeric(df_triaje["sex"], errors="coerce")
df_triaje["sex"] = df_triaje["sex"].astype("Int64")
df_triaje["sex"] = df_triaje["sex"].astype("category")

# Check the mapping
print("\nValue counts of 'sex' column after data conversion:")
df_triaje["sex"].value_counts(dropna=False)

# print(f"\nAfter excluding by non-valid sex values, the number of rows is: {df_triaje.shape[0]:,}")

In [ ]:
print(df_triaje["sex"].dtype)


## Time columns

In [ ]:
# List of time columns
time_columns = [
    "contact_entry_time",
    "contact_answer_time",
    "activation_time",
    "departure_time",
    "arrival_time",
    "load_time",
    "operative_time",
    "destination_time",
    "available_time"
]

for col in time_columns:
    df_triaje[col] = pd.to_datetime(
        df_triaje[col],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Count NaT values after conversion
print("\nCount of invalid or missing datetime values after conversion:")
for col in time_columns:
    nat_count = df_triaje[col].isna().sum()
    dtype = df_triaje[col].dtype
    print(f"{col} ({dtype}): {nat_count} NaT values")


## IDPROVINCIA - province

In [ ]:
# Original value counts
print("Value counts of 'province' column before conversion:")
df_triaje["province"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_province = {
    4 : "ALM",
    11: "CAD",
    14: "COR",
    18: "GRA",
    21: "HUE",
    23: "JAE",
    29: "MAL",
    41: "SEV"
}

# Map the values
df_triaje["province"] = df_triaje["province"].map(map_province)

# Check the mapping
print("Value counts of 'province' column after conversion:")
df_triaje["province"].value_counts(dropna=False)


## resource_dest_province_id

In [ ]:
# Original value counts
print("Value counts of 'resource_dest_province_id' column before conversion:")
df_triaje["resource_dest_province_id"].value_counts(dropna=False)


## IDALERTANTE - alert_receiver

This variable provides us with an identification code for the entity that received the alert. We have a total of 23 different IDs (as we will verify below). According to clinical indications, they will be regrouped into the following 6 groups:

- GROUP 1 '112': '112'.
- GROUP 2 'USER': 'USER' (User).
- GROUP 3 'POL/FG' (Police/Fire Brigade): '91', '92', '62', '80', '091', '092', '062', '080'.
- GROUP 4 'HS' (Health System): 'APS', 'SEU', 'SNU', 'HOS', '61', '061', 'ECA', 'PAC', 'DCC'.
- GROUP 5 'TELE' (Teleassistance): 'SAT'.
- GROUP 6 'OTHERS': 'PCI', 'OTR', 'SR', 'CRE', 'FAX', 'TEL', 'INA', 'RES'.

In [ ]:
# Original value counts
print("Value counts of 'alert_receiver' column before conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

In [ ]:
# Map raw alert receiver codes to grouped categories.
def map_alert_receiver(x):
    """
    This function:
    - Normalizes input values (strip spaces, uppercase).
    - Handles None/NaN safely.
    - Uses a dictionary-based mapping for maintainability.
    - Returns np.nan when the code is unknown or invalid.
    """

    # If null → return NaN
    if pd.isna(x):
        return np.nan

    # Normalize input (string only)
    x = str(x).strip().upper()
    if x == "":
        return np.nan

    # Dictionary of mappings
    mapping_dict = {
        # Group 1
        "112": "112",

        # Group 2
        "USU": "USER",

        # Group 3: POL/FG
        "91": "POL/FG", "92": "POL/FG", "62": "POL/FG", "80": "POL/FG",
        "091": "POL/FG", "092": "POL/FG", "062": "POL/FG", "080": "POL/FG",

        # Group 4: HS
        "APS": "HS", "SEU": "HS", "SNU": "HS", "HOS": "HS",
        "61": "HS", "061": "HS", "ECA": "HS", "PAC": "HS", "DCC": "HS",

        # Group 5: TELE
        "SAT": "TELE",

        # Group 6: OTHERS
        "PCI": "OTHERS", "OTR": "OTHERS", "SR": "OTHERS", "CRE": "OTHERS",
        "FAX": "OTHERS", "TEL": "OTHERS", "INA": "OTHERS", "RES": "OTHERS",
    }

    # Return mapped value or NaN if not found
    return mapping_dict.get(x, np.nan)


# Map the values
df_triaje["alert_receiver"] = df_triaje["alert_receiver"].apply(map_alert_receiver)

# Check the mapping
print("Value counts of 'alert_receiver' column after conversion:")
df_triaje["alert_receiver"].value_counts(dropna=False)

## IDESCENARIO - location_patient
- GROUP 1 'HO' (Home): 'DO'
- GROUP 2 'PR' (Public Road): 'VP'
- GROUP 3 'OTHERS': 'PI', 'CA', 'OT'.

In [ ]:
# Original value counts
print("Value counts of 'location_patient' column before conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

In [ ]:
# Map dictionary
map_location = {
    "DO": "HO",
    "VP": "PR",
    "CA": "OTHERS",
    "OT": "OTHERS",
    "PI": "OTHERS"
}

# Map the values
df_triaje["location_patient"] = df_triaje["location_patient"].map(map_location)

# Check the mapping
print("Value counts of 'location_patient' column after conversion:")
df_triaje["location_patient"].value_counts(dropna=False)

## IDTIPORECURSO - resource_type

In [ ]:
# Original value counts
print("Value counts of 'resource_type' column before conversion:")
df_triaje["resource_type"].value_counts(dropna=False)



In [ ]:
print(f"\nNumber of unique values in resource_type: {df_triaje['resource_type'].nunique()}")

## IDTARJETACORAZON - healthcard

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'healthcard' is: {df_triaje['healthcard'].isna().sum()}")
df_triaje["healthcard"] = df_triaje["healthcard"].notna().astype(int)
df_triaje["healthcard"].dtype

In [ ]:
df_triaje["healthcard"].value_counts(dropna = False)

## IDTARJETAAIRE - aircard

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

In [ ]:
print(f"Number of None values in 'aircard' is: {df_triaje['aircard'].isna().sum()}")
df_triaje["aircard"] = df_triaje["aircard"].notna().astype(int)
df_triaje["aircard"].dtype

In [ ]:
df_triaje["aircard"].value_counts(dropna = False)

## ESVULNERABLE - is_vulnerable

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

In [ ]:
df_triaje["is_vulnerable"] = df_triaje["is_vulnerable"].notna().astype(int)
df_triaje["is_vulnerable"].dtype

In [ ]:
df_triaje["is_vulnerable"].value_counts(dropna = False)

## Tests

In [ ]:
df_triaje.info()

## Transform data types

In [ ]:
df_triaje = kb.transform_column_dtypes(df_triaje)

In [ ]:
df_triaje.info()

In [ ]:
df_triaje.head(5)

# Creation of new variables

## Juicio clínico - icd

In [ ]:
cols_icd = ["icd1", "icd2", "icd3"]

df_triaje[cols_icd] = (
    df_triaje[cols_icd]
    .replace(r'^\s*$', pd.NA, regex=True)      # empty strings
    .replace(["nan", "NaN", "None", "none"], pd.NA)
    .astype("string")                          # enforce pandas string dtype (nullable)
)


In [ ]:
df_triaje.head(100)

In [ ]:
def combine_clean_columns(df, columns, sep):
    """
    Clean ICD-related columns and combine valid entries into a single string.
    Missing values are returned as pd.NA.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Input dataset.
    columns : list
        List of ICD columns to clean and combine.
    sep : str
        Separator used to join non-missing cleaned values.
    """

    def clean_value(x):
        """Normalize different missing representations to None."""
        if pd.isna(x):  # Detects real NaN/None/NaT
            return None
        if isinstance(x, str):
            x_strip = x.strip()
            # Remove empty strings and stringified missing markers
            if x_strip.lower() in ["", "nan", "none"]:
                return None
            return x_strip
        return x

    # Apply cleaning column-wise (avoid deprecated applymap)
    cleaned = df[columns].apply(lambda col: col.apply(clean_value))

    # Combine valid entries using the provided separator
    combined = cleaned.apply(
        lambda row: sep.join(str(x) for x in row if x is not None),
        axis=1
    )

    # Convert empty strings to real missing values
    combined = combined.replace("", pd.NA)

    return combined

df_triaje["icd_emerg_combined"] = combine_clean_columns(
    df_triaje,
    columns=["JUICIOCLINICO1", "JUICIOCLINICO2", "JUICIOCLINICO3"],
    sep=" | "       # You choose the separator manually
)


In [ ]:
# Remove '.' from the column
df_triaje["icd_emerg_combined"] = (
    df_triaje["icd_emerg_combined"]
    .astype("string")                # ensure pandas string dtype
    .str.replace(".", "", regex=False)
)



In [ ]:
df_triaje.info()
df_triaje['icd_emerg_combined'].value_counts(dropna=False)

In [ ]:
df_triaje.head(100)

## P1 real

In [ ]:
icd_table = pd.read_csv(os.path.join(source_tables_path, "icd/icd_p1outcomes.tsv"), sep='\t')

In [ ]:
icd_table.head(5)

In [ ]:
icd_table["AND/Exclusion"].value_counts()


In [ ]:
def process_icd_codes(df, icd_combined_col, reference_table):
    """
    Process combined ICD codes to detect exclusion rules and P1 priority outcomes.

    Parameters
    ----------
    df : pandas.DataFrame
        Main dataset containing the combined ICD codes column.
    icd_combined_col : str
        Name of the column containing ICD codes combined with a separator (e.g., " | ").
    reference_table : pandas.DataFrame
        Table containing ICD_Limpio, P1 outcomes, and AND/Exclusion fields.

    Returns
    -------
    pandas.DataFrame
        Filtered dataframe with a new column 'p1_flag' and exclusion rows removed.
    """

    # ===============================================================
    # 1. Normalize the reference ICD table and create lookup mappings
    # ===============================================================
    ref = reference_table.copy()

    # Flag P1 only when explicitly marked
    ref["p1_flag"] = ref["P1 outcomes"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().upper() == "P1" else 0
    )

    # Flag exclusions only when explicitly labeled as "Exclusion"
    ref["exclusion_flag"] = ref["AND/Exclusion"].apply(
        lambda x: 1 if isinstance(x, str) and x.strip().lower() == "exclusion" else 0
    )

    # Ensure ICD codes are strings to allow consistent matching
    ref["ICD_Limpio"] = ref["ICD_Limpio"].astype("string")

    # Create fast lookup dictionaries
    p1_map = dict(zip(ref["ICD_Limpio"], ref["p1_flag"]))
    exclusion_map = dict(zip(ref["ICD_Limpio"], ref["exclusion_flag"]))


    # ===============================================================
    # 2. Row-wise evaluation of ICD codes (safe split-based approach)
    # ===============================================================
    def evaluate_icd_row(text):
        """Evaluate exclusion or P1 priority for a single combined ICD code string."""
        
        if pd.isna(text):
            return {"exclude": False, "p1": 0}

        # Split codes by the separator and clean whitespace
        raw_codes = [c.strip() for c in str(text).split("|")]
        codes = [c for c in raw_codes if c != ""]

        # === Rule 1: exclude row if any ICD is explicitly marked as Exclusion ===
        for code in codes:
            if code in exclusion_map and exclusion_map[code] == 1:
                return {"exclude": True, "p1": None}

        # === Rule 2: assign P1 flag if any ICD in the row is labeled as P1 ===
        for code in codes:
            if code in p1_map and p1_map[code] == 1:
                return {"exclude": False, "p1": 1}

        # === Rule 3: otherwise, row is kept and marked as non-P1 ===
        return {"exclude": False, "p1": 0}


    # ===============================================================
    # 3. Apply evaluation logic to each row of the dataframe
    # ===============================================================
    evaluation = df[icd_combined_col].apply(evaluate_icd_row)

    df["p1_real"] = evaluation.apply(lambda r: r["p1"])
    df["exclude_row"] = evaluation.apply(lambda r: r["exclude"])


    # ===============================================================
    # 4. Remove excluded rows and clean the output dataframe
    # ===============================================================
    df_filtered = df[df["exclude_row"] == False].drop(columns=["exclude_row"])

    return df_filtered

# Call the function to process ICD codes
df_triaje = process_icd_codes(
    df=df_triaje,
    icd_combined_col="icd_emerg_combined",
    reference_table=icd_table
)

In [ ]:
df_triaje["p1_real"].value_counts(dropna=False)

In [ ]:
df_triaje[df_triaje["icd_emerg_combined"].notna()].head(100)

## Date variables

In [ ]:
df_edt_new_var = df_triaje.copy()

print(f"\nThe number of nan in PRIORIDAD is {df_edt_new_var['PRIORIDAD'].isnull().sum()}")

# P1_predicted is the priority predicted during the call, assigned by the system
# 1 is maximum priority, 0 is non-maximum priority
df_edt_new_var['P1_predicted'] = np.where(df_edt_new_var['PRIORIDAD']==1, 1, 0)

# Create the day of the week (number, Monday=0, Sunday=6) variable
df_edt_new_var['day_week'] = df_edt_new_var['FECHADEMANDA'].dt.dayofweek

# Create the month (number) variable
df_edt_new_var['month'] = df_edt_new_var['FECHADEMANDA'].dt.month

# Create the year variable
df_edt_new_var['year'] = df_edt_new_var['FECHADEMANDA'].dt.year

# Display the new columns and the original date column
print(df_edt_new_var[['FECHADEMANDA', 'day_week', 'month', 'year']].head())


## DATA QUALITY

In [ ]:
print(f"\nThe number of nan in IDTIPODEMANDA1 is {df_triaje['IDTIPODEMANDA1'].isnull().sum()}")

df_edt_new_var.head(50)
df_edt_new_var.head(100).to_csv('export_tables/triaje_table.csv', sep = ',', index=False)

In [ ]:
id_tarjeta_deseado = '81SNNM0DHR'  # Asegúrate de que el ID sea una cadena si así está en tu DataFrame

tabla_filtrada = df_edt_new_var[df_edt_new_var['IDTARJETASANITARIA'] == id_tarjeta_deseado]
tabla_filtrada.head(10)

In [ ]:
print(f"El NUHSA {id_tarjeta_deseado} aparece en el dataset {len(tabla_filtrada)} veces")

In [ ]:
print("IDTIPORECURSO", df_triaje['IDTIPORECURSO'].value_counts(dropna=False))